In [2]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

# 판다스 옵션으로 화면에 이쁘게 표현하기
pd.set_option('display.float_format', '{:.2f}'.format) # 소수점 두 자리까지만 표현

df = pd.read_csv('../sc_cust_info_txn_v1.5.csv')
df

,base_ym,dpro_tgt_perd_val,cust_ctg_type,cust_class,sex_type,age,efct_svc_count,dt_stop_yn,npay_yn,r3m_avg_bill_amt,r3m_A_avg_arpu_amt,r3m_B_avg_arpu_amt,r6m_A_avg_arpu_amt,r6m_B_avg_arpu_amt,termination_yn
0,202006,20200630,10001,C,F,28,0,N,N,2640.00,792.00,1584.00,0.00,0.00,Y
1,202006,20200630,10001,_,_,_,1,N,N,300.00,90.00,180.00,0.00,0.00,Y
2,202006,20200630,10001,E,F,24,1,N,N,16840.00,2526.00,6983.00,0.00,6981.00,N
3,202006,20200630,10001,F,F,32,1,N,N,15544.73,2331.71,6750.47,0.00,6508.80,N
4,202006,20200630,10001,D,M,18,1,N,N,4700.00,0.00,4502.00,0.00,4507.70,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9925,202006,20200630,10001,C,F,15,1,N,Y,1296.10,194.41,643.10,0.00,852.55,N
9926,202006,20200630,10001,G,M,12,1,N,N,13799.67,2069.95,10605.93,0.00,10603.93,N
9927,202006,20200630,10005,C,_,_,1,N,N,1396.20,1206.00,0.00,1212.00,0.00,N
9928,202006,20200630,10001,C,F,40,0,N,N,3140.00,942.00,1884.00,0.00,0.00,Y


In [3]:
# 탐색하기(확인하기)

## 데이터 구성과 특성 확인
df.info()

## 데이터의 기술적 통계 확인 (평균, 분산, 표준편차 등)
df.describe()

## 필요한 컬럼만 추출
# cust_class: 고객 등급
# sex_type: 성별
# age: 나이
# efct_svc_count: 사용 서비스 수
# dt_stop_yn: 서비스 중지 여부
# npay_yn: 미납 여부
# r3m_avg_bill_amt: 3개월 평균 요금
# r3m_A_avg_arpu_amt: A서비스 3개월 평균 요금
# r3m_B_avg_arpu_amt: B서비스 3개월 평균 요금
# termination_yn: 해지 여부
cust = df[ ["cust_class","sex_type","age","efct_svc_count","dt_stop_yn",
         "npay_yn","r3m_avg_bill_amt","r3m_A_avg_arpu_amt","r3m_B_avg_arpu_amt", "termination_yn"] ]

cust = cust.rename( columns = 
                   {
                       "cust_class" : 'class',          # 고객등급
                       "sex_type":'sex',                # 성별
                       "efct_svc_count":'service',      # 사용 서비스수
                       "dt_stop_yn":'stop',             # 서비스 중지 여부
                       "npay_yn":'npay',                # 미납 여부
                       "r3m_avg_bill_amt":'avg_bill',   # 3개월 평균 요금
                       "r3m_A_avg_arpu_amt":"A_bill",   # A서비스 3개월 평균요금
                       "r3m_B_avg_arpu_amt":'B_bill',   # B서비스 3개월 평균요금
                       "termination_yn":'termination'   # 해지 여부
                    }
                )

cust.describe()

<class 'pandas.DataFrame'>
RangeIndex: 9930 entries, 0 to 9929
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   base_ym             9930 non-null   int64  
 1   dpro_tgt_perd_val   9930 non-null   int64  
 2   cust_ctg_type       9930 non-null   int64  
 3   cust_class          9930 non-null   str    
 4   sex_type            9930 non-null   str    
 5   age                 9930 non-null   str    
 6   efct_svc_count      9930 non-null   int64  
 7   dt_stop_yn          9930 non-null   str    
 8   npay_yn             9930 non-null   str    
 9   r3m_avg_bill_amt    9930 non-null   float64
 10  r3m_A_avg_arpu_amt  9930 non-null   float64
 11  r3m_B_avg_arpu_amt  9930 non-null   float64
 12  r6m_A_avg_arpu_amt  9930 non-null   float64
 13  r6m_B_avg_arpu_amt  9930 non-null   float64
 14  termination_yn      9930 non-null   str    
dtypes: float64(5), int64(4), str(6)
memory usage: 1.1 MB


,service,avg_bill,A_bill,B_bill
count,9930.00,9930.00,9930.00,9930.00
mean,1.52,11817.74,1897.54,6395.26
std,15.40,139782.18,12353.42,83461.38
min,0.00,0.00,0.00,0.00
25%,1.00,3624.50,324.00,1260.00
50%,1.00,8284.47,1593.31,4768.62
75%,1.00,13720.00,2308.36,7982.00
max,905.00,12815682.13,1188998.30,7689409.28


In [ ]:
# 타입 변경하기
    # 데이터 호출 후 반드시 타입을 확인하는 것이 좋다.
    # 숫자형 데이터가 문자형으로 지정되어 있거나 그 반대의 경우 원하는 데이터 처리 결과가 도출되지 않을 수 있다.

## 1. 타입 확인 (증상 발견)
    # 10개의 컬럼으로 구성
    # age 컬럼이 str 문자열임을 확인
# cust.info()

## 2. 원인 확인
    # age 컬럼의 값 중 '_'가 있음
cust.head()

## (타입 변경 시도 (astype 함수))
    # astype으로 강제로 변경 시, 에러 발생
# cust['age'].astype('int') # 에러 발생, '_'는 숫자 0~9 중 어떤 것으로도 변경 불가

## 3. 결측치 대체 (replace)
    # '_' 문자값을 NaN 값으로 대체
    # 치환한 결과를 다시 cust 데이터프레임에 저장
# cust.dtypes으로 cust의 모든 컬럼이 category가 아닌 str, int, float임을 확인. map 대신 replace 사용
cust = cust.replace("_", np.nan)
# cust = cust.map(lambda x: np.nan if x == '_' else x)

## 4. 타입 변경 재시도
    # 특정 칼럼의 타입 변경하기
    # 예시: age 컬럼의 타입을 숫자형으로 변경
    # NaN의 경우 int type을 지원하지 않아 float type으로 변경
cust = cust.astype({'age': float})
# cust.info()

## 5. 원하는 행, 컬럼에 접근하기
cust.loc[[1, 2, 3], ['class', 'service', 'avg_bill']]

,class,service,avg_bill
1,NaN,1,300.00
2,E,1,16840.00
3,F,1,15544.73


In [6]:
# 결측치 처리
    # Missing Value로 데이터에 값이 없는 것을 의미
    # NA, Null, NaN으로 표시
    # 결측치 처리 시, copy 함수 사용할 것 (원본 데이터 손상 방지)
    # Run을 분리하지 않으면, 다다음 실습에서 전 실습이 반영되어버림
cust_fix = cust.copy() # 원본 데이터

In [7]:
# 실습을 위해 copy() 함수 복사
cust = cust_fix.copy()

In [ ]:
## 1. 결측치 직접 탐색
    # class, sex, age 컬럼에 NaN 결측치 확인
cust.head()

## 2. 결측치 간접 탐색 1
    # 전체 index: 9930 건
    # class 컬럼: 8529 건 non-null
    # sex 컬럼: 9249 건 non-null
    # age 컬럼: 9178 건 non-null
# cust.info()

## 3. 결측치 간접 탐색 2
    # class 컬럼: 1404 건
    # sex 컬럼: 681 건
    # age 컬럼: 752 건
cust.isnull().sum()

## ⭐️ 4.1. 결측치 처리 - 특정 값(15)으로 처리
    # fillna 함수로 NaN 값을 15로 변경
    # 실무에서는 편향될 수 있으므로 유의하여 사용
# fill_dict = { # Dictionary Comprehension
#     col: '15' if col in ['class', 'sex'] else 15
#     for col in cust.columns
# }
fill_dict = {}
for col in cust.columns:
    if col in ['class', 'sex']:
        fill_dict['col'] = '15'
    else:
        fill_dict['col'] = 15
cust = cust.fillna({ 'col': 15 })

## 4.2. 결측치 처리 - NaN 앞 인덱스로 처리
# cust = cust.ffill() # NaN 앞 인덱스의 데이터값으로 결측치 채우기

## 4.3. 결측치 처리 - NaN 뒤 인덱스로 처리
# cust = cust.bfill() # NaN 뒤 인덱스의 데이터값으로 결측치 채우기

## 4.4. 결측치 처리 - 중앙값으로 처리 (replace)
# median = cust[ 'age' ].median()
# cust['age'] = cust['age'].replace(np.nan, median)

## 4.5. 결측치 처리 - 제거하기
    # 채우기로 처리할 경우 왜곡이 발생할 수 있다.
    # 이때는 제거하는 기법으로 데이터의 정합성을 보존할 수 있다.
    # how: 처리 방식
        # listwise 방식: record 항목 중 1개의 값이라도 NaN이면 해당 행 제거
        # pairwise 방식: record 항목 전체가 NaN이면 해당 행 제거
    # subset: 특정 컬럼에서 NaN이 존재할 경우, 해당 인덱스를 삭제
# cust = cust.dropna() # default - listwise 방식, 9930 건에서 8228 건으로 감소
# cust = cust.dropna(how='all') # pairwise 방식, 9930 건 유지, 모든 칼럼이 NaN 값인 인덱스 없음.
cust = cust.dropna(subset=['class', 'sex', 'age']) # subset에 있는 값에 NaN이 있을 경우, 해당 열 제거


## 5. 결측치 제거 결과 확인
cust.info()
cust.isnull().sum()
cust.head()

<class 'pandas.DataFrame'>
Index: 8228 entries, 0 to 9929
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   class        8228 non-null   str    
 1   sex          8228 non-null   str    
 2   age          8228 non-null   float64
 3   service      8228 non-null   int64  
 4   stop         8228 non-null   str    
 5   npay         8228 non-null   str    
 6   avg_bill     8228 non-null   float64
 7   A_bill       8228 non-null   float64
 8   B_bill       8228 non-null   float64
 9   termination  8228 non-null   str    
dtypes: float64(4), int64(1), str(5)
memory usage: 707.1 KB


,class,sex,age,service,stop,npay,avg_bill,A_bill,B_bill,termination
0,C,F,28.00,0,N,N,2640.00,792.00,1584.00,Y
2,E,F,24.00,1,N,N,16840.00,2526.00,6983.00,N
3,F,F,32.00,1,N,N,15544.73,2331.71,6750.47,N
4,D,M,18.00,1,N,N,4700.00,0.00,4502.00,N
5,C,F,78.00,1,N,N,1361.80,1174.00,0.00,N


In [ ]:
# 결측치 처리한 데이터프레임 파일로 저장하기

## 1. 인덱스 초기화(새롭게 인덱스 생성)
    # drop=True: 이빨 빠진 기존 인덱스를 버리도록 옵션 설정
    # inplace=True: 해당 결과를 다시 cust 데이터프레임에 적용
cust.reset_index(drop=True, inplace=True)
cust

## 2. 결측치 제거 결과 확인
cust.isna().sum()

## 2. csv 파일로 저장
    # index=False: 인덱스 칼럼 없이 저장 (실무에서 주로 사용)
# cust.to_csv('cust_data2.csv', index=False)

class          0
sex            0
age            0
service        0
stop           0
npay           0
avg_bill       0
A_bill         0
B_bill         0
termination    0
dtype: int64